# 03 – RQ3: Interrupted Time Series Analysis

**Research Question:** Did the frequency and proportion of enterprise AI incidents change significantly after the EU AI Act proposal (April 2021) and the U.S. Executive Order on AI Safety (October 2023)?

**Hypothesis:** The proportion of unintentional enterprise incidents decreased significantly following both regulatory milestones.

**Method:** Segmented regression (Interrupted Time Series) on quarterly incident counts.  
Newey-West standard errors correct for autocorrelation in the time series.

**Intervention points:**
- Intervention 1: Q2 2021 (EU AI Act proposal, April 2021)
- Intervention 2: Q4 2023 (U.S. Executive Order on AI Safety, October 2023)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import statsmodels.api as sm
from pathlib import Path

FIGURES_DIR = Path('../outputs/figures')
df = pd.read_pickle('../outputs/incidents_prepared.pkl')

In [ ]:
# Identify date/year columns
date_col = [c for c in df.columns if 'date' in c.lower()][0]
df['date_parsed'] = pd.to_datetime(df[date_col], errors='coerce')
df = df.dropna(subset=['date_parsed'])

# Restrict to incidents from 2014 onwards (sparse data before this)
df = df[df['date_parsed'].dt.year >= 2014].copy()

# Create a quarter period column
df['quarter'] = df['date_parsed'].dt.to_period('Q')
print(f'Date range: {df["date_parsed"].min().date()} to {df["date_parsed"].max().date()}')
print(f'Total incidents in range: {len(df):,}')

In [ ]:
# Aggregate to quarterly counts
quarterly = df.groupby('quarter').size().reset_index(name='incident_count')
quarterly['quarter_str'] = quarterly['quarter'].astype(str)

# Build a continuous time index (T)
quarterly['T'] = range(len(quarterly))

# --- Intervention 1: Q2 2021 (EU AI Act proposal) ---
INT1 = pd.Period('2021Q2', freq='Q')
quarterly['X1'] = (quarterly['quarter'] >= INT1).astype(int)

# --- Intervention 2: Q4 2023 (U.S. Executive Order) ---
INT2 = pd.Period('2023Q4', freq='Q')
quarterly['X2'] = (quarterly['quarter'] >= INT2).astype(int)

# Time-since-intervention variables (slope change terms)
int1_idx = quarterly[quarterly['quarter'] >= INT1].index.min()
int2_idx = quarterly[quarterly['quarter'] >= INT2].index.min()

quarterly['T_X1'] = np.where(quarterly['X1'] == 1,
                              quarterly['T'] - quarterly.loc[int1_idx, 'T'],
                              0)
quarterly['T_X2'] = np.where(quarterly['X2'] == 1,
                              quarterly['T'] - quarterly.loc[int2_idx, 'T'],
                              0)

print(quarterly[['quarter_str', 'incident_count', 'T', 'X1', 'X2', 'T_X1', 'T_X2']].head(10))

In [ ]:
# Segmented regression with two intervention points
# Model: Y_t = B0 + B1*T + B2*X1 + B3*(T*X1) + B4*X2 + B5*(T*X2) + e_t

y = quarterly['incident_count']
X = sm.add_constant(quarterly[['T', 'X1', 'T_X1', 'X2', 'T_X2']])

# OLS fit with Newey-West HAC standard errors (corrects for autocorrelation)
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(model.summary())

In [ ]:
# Extract key parameters
params = model.params
pvals  = model.pvalues

print('\nParameter Interpretation:')
print(f'  B0 (Baseline level at T=0):         {params["const"]:+.3f}  (p={pvals["const"]:.4f})')
print(f'  B1 (Pre-intervention trend):         {params["T"]:+.3f}  (p={pvals["T"]:.4f})')
print(f'  B2 (Level change at EU AI Act):      {params["X1"]:+.3f}  (p={pvals["X1"]:.4f})')
print(f'  B3 (Trend change after EU AI Act):   {params["T_X1"]:+.3f}  (p={pvals["T_X1"]:.4f})')
print(f'  B4 (Level change at U.S. EO):        {params["X2"]:+.3f}  (p={pvals["X2"]:.4f})')
print(f'  B5 (Trend change after U.S. EO):     {params["T_X2"]:+.3f}  (p={pvals["T_X2"]:.4f})')

In [ ]:
# Plot: observed counts + fitted regression line with intervention markers
quarterly['fitted'] = model.fittedvalues

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(quarterly['T'], quarterly['incident_count'],
       color='steelblue', alpha=0.5, label='Observed quarterly count')
ax.plot(quarterly['T'], quarterly['fitted'],
        color='crimson', linewidth=2, label='Fitted segmented regression')

# Mark intervention points
for idx, label, color in [
    (int1_idx, 'EU AI Act (Q2 2021)', 'darkorange'),
    (int2_idx, 'U.S. EO (Q4 2023)',   'green'),
]:
    ax.axvline(quarterly.loc[idx, 'T'], color=color, linestyle='--', linewidth=1.5, label=label)

# X-axis labels: show every 4 quarters
tick_positions = quarterly['T'][::4].tolist()
tick_labels    = quarterly['quarter_str'][::4].tolist()
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)

ax.set_title('Interrupted Time Series: AI Incident Counts with Regulatory Interventions',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Quarter')
ax.set_ylabel('Number of Incidents')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq3_interrupted_time_series.png', dpi=150)
plt.show()